# 📊 Analyse Exploratoire des Données (EDA)

## Objectif
Explorer le dataset Kaggle "Influencer or Observer" pour comprendre:
- La distribution des classes (Influencers vs Observers)
- Les caractéristiques des tweets
- Les patterns textuels
- Les features utilisateur importantes

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Modules locaux
from src.data_loader import load_training_data, load_test_data
from src.preprocessing import extract_full_text, clean_text
from src.feature_engineering import count_hashtags, count_mentions, count_emojis

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Chargement des Données

In [ ]:
# Charger les données d'entraînement
print("📥 Chargement des données...")
df_train = load_training_data('../data/train.jsonl')
print(f"✓ Données chargées: {len(df_train)} tweets")
print(f"✓ Nombre d'utilisateurs uniques: {df_train['challenge_id'].nunique()}")

# Aperçu
df_train.head()

## 2. Distribution des Classes

In [ ]:
# Distribution des labels (par utilisateur)
user_labels = df_train.groupby('challenge_id')['label'].first()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
user_labels.value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('Distribution des Classes (Utilisateurs)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label (0=Observer, 1=Influencer)')
axes[0].set_ylabel('Nombre d\'utilisateurs')
axes[0].set_xticklabels(['Observer (0)', 'Influencer (1)'], rotation=0)

# Pie chart
user_labels.value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['skyblue', 'salmon'])
axes[1].set_title('Proportion des Classes', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\n📊 Statistiques:")
print(user_labels.value_counts())
print(f"\n⚖️ Ratio Influencers/Observers: {user_labels.value_counts()[1] / user_labels.value_counts()[0]:.2f}")

## 3. Analyse Textuelle

In [ ]:
# Extraire le texte complet
print("📝 Extraction du texte...")
df_train['full_text_extracted'] = df_train.apply(extract_full_text, axis=1)

# Statistiques de longueur
df_train['text_length'] = df_train['full_text_extracted'].str.len()
df_train['word_count'] = df_train['full_text_extracted'].str.split().str.len()

print("✓ Texte extrait")
df_train[['full_text_extracted', 'text_length', 'word_count', 'label']].head()

In [ ]:
# Distribution de la longueur des tweets par classe
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Longueur de caractères
df_train[df_train['label'] == 0]['text_length'].hist(bins=50, alpha=0.6, label='Observer', ax=axes[0], color='skyblue')
df_train[df_train['label'] == 1]['text_length'].hist(bins=50, alpha=0.6, label='Influencer', ax=axes[0], color='salmon')
axes[0].set_title('Distribution de la Longueur des Tweets (caractères)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Longueur (caractères)')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

# Nombre de mots
df_train[df_train['label'] == 0]['word_count'].hist(bins=50, alpha=0.6, label='Observer', ax=axes[1], color='skyblue')
df_train[df_train['label'] == 1]['word_count'].hist(bins=50, alpha=0.6, label='Influencer', ax=axes[1], color='salmon')
axes[1].set_title('Distribution du Nombre de Mots', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nombre de mots')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

plt.tight_layout()
plt.show()

# Statistiques
print("\n📊 Statistiques de longueur par classe:")
print(df_train.groupby('label')[['text_length', 'word_count']].describe())

## 4. Features Textuelles

In [ ]:
# Extraire les features
print("🔧 Extraction des features...")
df_train['num_hashtags'] = df_train['full_text_extracted'].apply(count_hashtags)
df_train['num_mentions'] = df_train['full_text_extracted'].apply(count_mentions)
df_train['num_emojis'] = df_train['full_text_extracted'].apply(count_emojis)

print("✓ Features extraites")
df_train[['full_text_extracted', 'num_hashtags', 'num_mentions', 'num_emojis', 'label']].head(10)

In [ ]:
# Distribution des features par classe
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

features = ['num_hashtags', 'num_mentions', 'num_emojis']
titles = ['Hashtags', 'Mentions', 'Emojis']

for i, (feature, title) in enumerate(zip(features, titles)):
    df_train.groupby('label')[feature].mean().plot(kind='bar', ax=axes[i], color=['skyblue', 'salmon'])
    axes[i].set_title(f'Moyenne de {title} par Classe', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Label')
    axes[i].set_ylabel(f'Nombre moyen de {title.lower()}')
    axes[i].set_xticklabels(['Observer (0)', 'Influencer (1)'], rotation=0)

plt.tight_layout()
plt.show()

print("\n📊 Moyennes par classe:")
print(df_train.groupby('label')[features].mean())

## 5. Mots les Plus Fréquents

In [ ]:
# Importer NLTK stopwords
from nltk.corpus import stopwords
import nltk

# Télécharger si nécessaire
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

french_stopwords = set(stopwords.words('french'))
print(f"✓ Stopwords français chargés: {len(french_stopwords)} mots")

In [ ]:
def get_top_words(texts, n=20, remove_stopwords=True):
    """Récupère les n mots les plus fréquents"""
    all_words = []
    for text in texts:
        if pd.notna(text):
            words = text.lower().split()
            if remove_stopwords:
                words = [w for w in words if w not in french_stopwords and len(w) > 2]
            all_words.extend(words)
    return Counter(all_words).most_common(n)

# Top mots pour chaque classe
observers_text = df_train[df_train['label'] == 0]['full_text_extracted']
influencers_text = df_train[df_train['label'] == 1]['full_text_extracted']

top_observers = get_top_words(observers_text, n=20)
top_influencers = get_top_words(influencers_text, n=20)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Observers
words, counts = zip(*top_observers)
axes[0].barh(range(len(words)), counts, color='skyblue')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].invert_yaxis()
axes[0].set_title('Top 20 Mots - Observers (0)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Fréquence')

# Influencers
words, counts = zip(*top_influencers)
axes[1].barh(range(len(words)), counts, color='salmon')
axes[1].set_yticks(range(len(words)))
axes[1].set_yticklabels(words)
axes[1].invert_yaxis()
axes[1].set_title('Top 20 Mots - Influencers (1)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Fréquence')

plt.tight_layout()
plt.show()

## 6. Analyse des Features Utilisateur

In [ ]:
# Colonnes user
user_cols = [col for col in df_train.columns if col.startswith('user.')]
print(f"📊 {len(user_cols)} colonnes utilisateur trouvées")
print("\nExemples:")
print(user_cols[:10])

In [ ]:
# Agréger par utilisateur
user_df = df_train.groupby('challenge_id').agg({
    'label': 'first',
    'user.statuses_count': 'first',
    'user.followers_count': 'first',
    'user.friends_count': 'first',
    'user.listed_count': 'first',
    'user.favourites_count': 'first'
}).reset_index()

# Calculer le ratio followers/friends
user_df['followers_friends_ratio'] = user_df['user.followers_count'] / (user_df['user.friends_count'] + 1)

print("✓ Données utilisateur agrégées")
user_df.head()

In [ ]:
# Distribution du ratio followers/friends
fig, ax = plt.subplots(figsize=(12, 6))

user_df[user_df['label'] == 0]['followers_friends_ratio'].hist(
    bins=50, alpha=0.6, label='Observer', ax=ax, color='skyblue', range=(0, 10)
)
user_df[user_df['label'] == 1]['followers_friends_ratio'].hist(
    bins=50, alpha=0.6, label='Influencer', ax=ax, color='salmon', range=(0, 10)
)

ax.set_title('Distribution du Ratio Followers/Friends', fontsize=14, fontweight='bold')
ax.set_xlabel('Ratio Followers/Friends')
ax.set_ylabel('Fréquence')
ax.legend()
ax.set_xlim(0, 10)

plt.tight_layout()
plt.show()

print("\n📊 Statistiques du ratio:")
print(user_df.groupby('label')['followers_friends_ratio'].describe())

## 7. Conclusions de l'EDA

### Observations clés:
1. **Distribution des classes**: Vérifier si les classes sont équilibrées ou déséquilibrées
2. **Longueur des tweets**: Y a-t-il des différences entre Influencers et Observers?
3. **Features textuelles**: Les Influencers utilisent-ils plus de hashtags/mentions?
4. **Vocabulaire**: Quels mots sont caractéristiques de chaque classe?
5. **Ratio followers/friends**: Métrique discriminante importante (mais pas disponible au test!)

### Prochaines étapes:
- Tester des modèles baseline (Dummy, Logistic Regression)
- Feature engineering avancé
- Modèles plus sophistiqués (Random Forest, XGBoost, Transformers)
- Optimisation des hyperparamètres

In [ ]:
# Sauvegarder le notebook pour références futures
print("✅ EDA terminée!")
print("📝 Passez au notebook 02_baseline.ipynb pour les premiers modèles")